In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import joblib


In [ ]:
# Substitua os caminhos abaixo pelos seus arquivos de dados
train_df = pd.read_csv("Base_Treino_Consolidada.csv")
test_df = pd.read_csv("Base_Teste_Consolidada.csv")
train_df.head()
test_df.head()


,order_item_id,shipping_limit_date,price,freight_value,seller_city,seller_state,product_weight_g,order_status,order_purchase_timestamp,order_approved_at,...,geolocation_state_customer,payment_value,product_volume,geolocation_city_seller,geolocation_state_seller,distancia_km,purchase_to_approval_days,approval_to_carrier_days,carrier_to_customer_days,categoria_principal
0,1,2017-09-19 09:45:35,58.9,13.29,volta redonda,SP,650.0,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,...,RJ,72.19,3528.0,volta redonda,RJ,301.005664,0.0,6.0,1.0,Esportes e Lazer
1,1,2017-05-03 11:05:13,239.9,19.93,sao paulo,SP,30000.0,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,...,SP,259.83,60000.0,sao paulo,SP,589.274140,0.0,8.0,8.0,Outros e Indústria
2,1,2018-01-18 14:48:30,199.0,17.87,borda da mata,MG,3050.0,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,...,MG,216.87,14157.0,borda da mata,MG,312.495046,0.0,1.0,6.0,Casa e Decoração
3,1,2017-02-13 13:57:51,199.9,18.14,loanda,PR,3750.0,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,...,SP,218.04,42000.0,loanda,PR,646.221788,0.0,11.0,13.0,Ferramentas e Construção
4,1,2017-05-23 03:55:27,21.9,12.69,ribeirao preto,SP,450.0,delivered,2017-05-15 21:42:34,2017-05-17 03:55:27,...,MG,34.59,2880.0,ribeirao preto,SP,161.597746,1.0,0.0,5.0,NaN


In [8]:
label_encoder = LabelEncoder()
train_df['seller_city'] = label_encoder.fit_transform(train_df['seller_city'])
train_df['seller_state'] = label_encoder.fit_transform(train_df['seller_state'])
train_df['customer_city'] = label_encoder.fit_transform(train_df['customer_city'])
train_df['customer_state'] = label_encoder.fit_transform(train_df['customer_state'])


In [9]:
features = [
    'freight_value', 'product_weight_g', 'payment_value', 'product_volume',
    'distancia_km', 'purchase_to_approval_days',
    'approval_to_carrier_days', 'carrier_to_customer_days',
]
target = 'delivery_time_model (days)'

X = train_df[features]
y = train_df[target]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
model = xgb.XGBRegressor(
    n_estimators=100,  # Número de árvores
    learning_rate=0.1,  # Taxa de aprendizado
    max_depth=5,  # Profundidade máxima das árvores
    random_state=42
)
model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [12]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")


Mean Squared Error: 34.25317009261544


In [ ]:
label_encoder = LabelEncoder()

# Codificar as colunas categóricas, como seller_state ou customer_state
test_df['seller_state'] = label_encoder.fit_transform(test_df['seller_state'])
test_df['customer_state'] = label_encoder.fit_transform(test_df['customer_state'])
test_df['seller_city'] = label_encoder.transform(test_df['seller_city'])
test_df['customer_city'] = label_encoder.fit_transform(test_df['customer_city'])


# Selecionar os recursos para a base de teste
X_new = test_df[features]
y_new_pred = model.predict(X_new)

# Salvar ou exibir os resultados
test_df['predicted_delivery_time'] = y_new_pred
test_df[['order_id', 'predicted_delivery_time']].to_csv('predicted_delivery_times.csv', index=False)
print("Previsões salvas em 'predicted_delivery_times.csv'")


KeyError: 'customer_state'